# (*≧ω≦*) OchreML: Fast Classical Machine Learning in Rust with PyO3 (＾▽＾)
### A Complete Tour with Scikit-Learn Benchmarking, Visualizations, and Multi-Format Interoperability

Welcome to the official demonstration notebook for **OchreML**! (*^▽^*)  
OchreML is a high-performance classical machine learning library engineered in Rust, featuring zero-overhead PyO3 bindings and standard Scikit-Learn API compatibility.

---

### Highlights of OchreML (o´∀｀o)
- **High-Performance Rust Kernels:** Outperforms Scikit-Learn on Decision Tree training and prediction!
- **Linear Regression (OLS / Normal Equation):** Closed-form analytical solution with automated Tikhonov (Ridge) regularization fallback.
- **Decision Tree Classifier & Regressor:** CART algorithm with $O(D \cdot N \log N)$ sliding split search and $O(1)$ MSE updates.
- **Universal Multi-Format Interoperability:** Native zero-copy support for **NumPy**, **Pandas**, **Polars**, and **standard Python lists**.
- **Scikit-Learn Drop-In Interface:** Exact same `fit()`, `predict()`, `score()`, `coef_`, and `intercept_` attributes.
- **Pure Kaomoji Documentation:** Friendly, expressive, and strictly zero Unicode emojis (*≧ω≦*).


## 1. Installation & Environment Setup (o´∀｀o)

Install OchreML and verify imports alongside Scikit-Learn, Pandas, Polars, and Matplotlib:


In [ ]:
# Install OchreML (from PyPI)
# !pip install ochreml

import time
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

# Import OchreML models
import ochreml
from ochreml import (
    LinearRegression as OchreLinearRegression,
    DecisionTreeClassifier as OchreDecisionTreeClassifier,
    DecisionTreeRegressor as OchreDecisionTreeRegressor,
)

# Import Scikit-Learn reference models
from sklearn.linear_model import LinearRegression as SklearnLinearRegression
from sklearn.tree import (
    DecisionTreeClassifier as SklearnDecisionTreeClassifier,
    DecisionTreeRegressor as SklearnDecisionTreeRegressor,
)
from sklearn.metrics import r2_score, accuracy_score

print("OchreML loaded successfully! (*≧ω≦*)")
print("OchreML Version:", ochreml.__version__)


---
## 2. Linear Regression: OchreML vs Scikit-Learn (*^▽^*)

OchreML solves Ordinary Least Squares analytically using the Normal Equation:
$$\theta = (X^T X)^{-1} X^T y$$

Its internal Rust engine accumulates the symmetric Gram matrix in a single pass without allocating transpose matrices. If the matrix is ill-conditioned or collinear, Tikhonov regularization is engaged automatically to guarantee numerical stability! (＾▽＾)

Let's fit both **OchreML** and **Scikit-Learn** side-by-side on a synthetic dataset with noise:


In [ ]:
# Generate synthetic 1D linear regression dataset
np.random.seed(42)
n_samples = 80
X_raw = np.linspace(0.0, 10.0, n_samples)
true_slope = 3.2
true_intercept = 4.5
noise = np.random.normal(0.0, 2.0, n_samples)
y_raw = true_slope * X_raw + true_intercept + noise

df_train = pd.DataFrame({"feature_x": X_raw})
y_train = pd.Series(y_raw, name="target_y")

# 1. Fit OchreML LinearRegression
ochre_lr = OchreLinearRegression(fit_intercept=True)
ochre_lr.fit(df_train, y_train)

# 2. Fit Scikit-Learn LinearRegression
sk_lr = SklearnLinearRegression(fit_intercept=True)
sk_lr.fit(df_train, y_train)

# Compare fitted parameters
print("--- Parameter Comparison ---")
print(f"OchreML Coef:     {ochre_lr.coef_[0]:.6f} | Intercept: {ochre_lr.intercept_:.6f}")
print(f"Sklearn Coef:     {sk_lr.coef_[0]:.6f} | Intercept: {sk_lr.intercept_:.6f}")
print(f"OchreML R^2:      {ochre_lr.score(df_train, y_train):.6f}")
print(f"Sklearn R^2:      {sk_lr.score(df_train, y_train):.6f}")
print(f"Max Coef Diff:    {abs(ochre_lr.coef_[0] - sk_lr.coef_[0]):.2e} (*≧ω≦*)")


### Visualizing the Linear Regression Fit (o´∀｀o)
Both regression lines overlap perfectly on top of the observed data points:


In [ ]:
x_grid = np.linspace(0.0, 10.0, 200)
x_grid_df = pd.DataFrame({"feature_x": x_grid})
ochre_line = ochre_lr.predict(x_grid_df)
sk_line = sk_lr.predict(x_grid_df)

plt.figure(figsize=(9, 5))
plt.scatter(X_raw, y_raw, color="#3498db", edgecolors="black", alpha=0.75, label="Observed Samples")
plt.plot(x_grid, ochre_line, color="#e74c3c", linewidth=2.5, label="OchreML OLS Fit")
plt.plot(x_grid, sk_line, color="#2ecc71", linestyle="--", linewidth=2.0, label="Scikit-Learn Fit")
plt.title("Linear Regression: OchreML vs Scikit-Learn (＾▽＾)", fontsize=13, fontweight="bold")
plt.xlabel("Feature X")
plt.ylabel("Target y")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(frameon=True)
plt.show()


---
## 3. Decision Tree Classifier: OchreML vs Scikit-Learn (*^▽^*)

OchreML's CART classification tree evaluates splits in $O(D \cdot N \log N)$ time by sliding class histograms along sorted feature dimensions.

Let's test both classifiers on a 2D non-linear classification dataset:


In [ ]:
# Generate non-linear 2D classification dataset (concentric clusters)
np.random.seed(1337)
n_pts = 160

# Class 0: inner core
r0 = np.random.uniform(0.5, 2.0, n_pts // 2)
t0 = np.random.uniform(0, 2 * np.pi, n_pts // 2)
x0 = r0 * np.cos(t0)
y0 = r0 * np.sin(t0)

# Class 1: outer shell
r1 = np.random.uniform(3.0, 5.0, n_pts // 2)
t1 = np.random.uniform(0, 2 * np.pi, n_pts // 2)
x1 = r1 * np.cos(t1)
y1 = r1 * np.sin(t1)

X_cls = np.vstack([np.column_stack([x0, y0]), np.column_stack([x1, y1])])
y_cls = np.array([0.0] * (n_pts // 2) + [1.0] * (n_pts // 2))

# Train-test split (80% train, 20% test)
indices = np.random.permutation(n_pts)
train_idx, test_idx = indices[:int(0.8 * n_pts)], indices[int(0.8 * n_pts):]

X_tr, y_tr = X_cls[train_idx], y_cls[train_idx]
X_te, y_te = X_cls[test_idx], y_cls[test_idx]

# 1. Fit OchreML DecisionTreeClassifier
ochre_tree = OchreDecisionTreeClassifier(criterion="gini", max_depth=4)
ochre_tree.fit(X_tr, y_tr)

# 2. Fit Scikit-Learn DecisionTreeClassifier
sk_tree = SklearnDecisionTreeClassifier(criterion="gini", max_depth=4, random_state=42)
sk_tree.fit(X_tr, y_tr)

ochre_acc = ochre_tree.score(X_te, y_te)
sk_acc = accuracy_score(y_te, sk_tree.predict(X_te))

print(f"OchreML Test Accuracy:     {ochre_acc * 100:.2f}% (＾▽＾)")
print(f"Scikit-Learn Test Accuracy: {sk_acc * 100:.2f}% (*≧ω≦*)")
print("Discovered Classes:", ochre_tree.classes_)


### Side-by-Side Decision Boundary Visualizations (*^▽^*)


In [ ]:
# Create dense 2D evaluation grid
x_min, x_max = X_cls[:, 0].min() - 1, X_cls[:, 0].max() + 1
y_min, y_max = X_cls[:, 1].min() - 1, X_cls[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 150), np.linspace(y_min, y_max, 150))
grid_points = np.c_[xx.ravel(), yy.ravel()]

Z_ochre = np.array(ochre_tree.predict(grid_points)).reshape(xx.shape)
Z_sk = sk_tree.predict(grid_points).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].contourf(xx, yy, Z_ochre, alpha=0.35, cmap="coolwarm")
axes[0].scatter(X_cls[:, 0], X_cls[:, 1], c=y_cls, cmap="coolwarm", edgecolors="k", s=35)
axes[0].set_title(f"OchreML Decision Tree (Acc: {ochre_acc * 100:.1f}%) (*≧ω≦*)", fontweight="bold")
axes[0].set_xlabel("Feature X1")
axes[0].set_ylabel("Feature X2")
axes[0].grid(True, linestyle="--", alpha=0.4)

axes[1].contourf(xx, yy, Z_sk, alpha=0.35, cmap="coolwarm")
axes[1].scatter(X_cls[:, 0], X_cls[:, 1], c=y_cls, cmap="coolwarm", edgecolors="k", s=35)
axes[1].set_title(f"Scikit-Learn Decision Tree (Acc: {sk_acc * 100:.1f}%) (*^▽^*)", fontweight="bold")
axes[1].set_xlabel("Feature X1")
axes[1].set_ylabel("Feature X2")
axes[1].grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()


---
## 4. Decision Tree Regressor: OchreML vs Scikit-Learn (o´∀｀o)

`DecisionTreeRegressor` minimizes Mean Squared Error (MSE / Variance) across splits. OchreML maintains running sums of $y$ and $y^2$ to compute variance reductions in $O(1)$ constant time per split candidate.

Let's test both regressors on a non-linear sine wave dataset:


In [ ]:
# Generate non-linear sine wave dataset
np.random.seed(99)
X_sine = np.sort(np.random.uniform(0.0, 2 * np.pi, 100)).reshape(-1, 1)
y_sine = np.sin(X_sine.ravel()) + np.random.normal(0.0, 0.15, 100)

# Fit OchreML DecisionTreeRegressor
ochre_dtr = OchreDecisionTreeRegressor(criterion="squared_error", max_depth=4)
ochre_dtr.fit(X_sine, y_sine)

# Fit Scikit-Learn DecisionTreeRegressor
sk_dtr = SklearnDecisionTreeRegressor(criterion="squared_error", max_depth=4, random_state=42)
sk_dtr.fit(X_sine, y_sine)

print(f"OchreML R^2 Score:     {ochre_dtr.score(X_sine, y_sine):.4f} (o´∀｀o)")
print(f"Scikit-Learn R^2 Score: {sk_dtr.score(X_sine, y_sine):.4f} (＾▽＾)")


### Comparing Non-Linear Step Predictions (*^▽^*)


In [ ]:
X_eval = np.linspace(0.0, 2 * np.pi, 300).reshape(-1, 1)
y_ochre_pred = ochre_dtr.predict(X_eval)
y_sk_pred = sk_dtr.predict(X_eval)

plt.figure(figsize=(10, 5))
plt.scatter(X_sine, y_sine, s=30, edgecolor="black", c="navy", alpha=0.6, label="Training Samples")
plt.plot(X_eval, np.sin(X_eval), color="black", linestyle=":", label="Ground Truth sin(x)")
plt.plot(X_eval, y_ochre_pred, color="#e67e22", linewidth=2.5, label="OchreML Regressor (max_depth=4)")
plt.plot(X_eval, y_sk_pred, color="#2980b9", linestyle="--", linewidth=2.0, label="Scikit-Learn Regressor")
plt.title("Decision Tree Regression: OchreML vs Scikit-Learn (o´∀｀o)", fontsize=13, fontweight="bold")
plt.xlabel("X")
plt.ylabel("Target y")
plt.legend(frameon=True)
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()


---
## 5. Universal Multi-Format Interoperability (≧∇≦)/

OchreML works seamlessly with all primary data science containers without requiring manual format conversions:


In [ ]:
# 1. Train model using Pandas DataFrame & Series
p_df = pd.DataFrame({"x1": [1.0, 2.0, 3.0, 4.0], "x2": [2.0, 1.0, 4.0, 3.0]})
p_target = pd.Series([10.0, 20.0, 30.0, 40.0])
interop_model = OchreLinearRegression()
interop_model.fit(p_df, p_target)

# 2. Predict using Polars DataFrame
pl_query = pl.DataFrame({"x1": [2.5, 3.5], "x2": [1.5, 3.5]})
print("Prediction from Polars:", interop_model.predict(pl_query), "(*≧ω≦*)")

# 3. Predict using Python standard nested lists
list_query = [[1.0, 2.0], [4.0, 3.0]]
print("Prediction from Lists:", interop_model.predict(list_query), "(o´∀｀o)")

# 4. Predict using NumPy ndarray
np_query = np.array([[2.0, 1.0]])
print("Prediction from NumPy:", interop_model.predict(np_query), "(＾▽＾)")


---
## 6. Comprehensive Performance Benchmark Suite (*≧ω≦*)

Here we run an automated benchmark comparing **OchreML (pure Rust kernel)** against **Scikit-Learn (Cython backend)** on larger synthetic datasets. We measure:
1. **Fit Time (ms):** Model training latency.
2. **Predict Time (ms):** Batch inference latency.
3. **Accuracy / $R^2$ Score:** Metric parity.


In [ ]:
# Benchmark Function
def benchmark_suite():
    results = []

    # 1. Decision Tree Classifier (Gini, max_depth=5)
    n_samples, n_features = 2500, 8
    np.random.seed(1337)
    X = np.random.randn(n_samples, n_features)
    y = ((X[:, 0] * 2.0 + X[:, 1] ** 2 - X[:, 2]) > 0.5).astype(float)
    X_train, X_test = X[:2000], X[2000:]
    y_train, y_test = y[:2000], y[2000:]

    # OchreML
    t0 = time.perf_counter()
    ochre_c = OchreDecisionTreeClassifier(criterion="gini", max_depth=5)
    ochre_c.fit(X_train, y_train)
    ochre_c_fit = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    ochre_c_preds = ochre_c.predict(X_test)
    ochre_c_pred = (time.perf_counter() - t0) * 1000
    ochre_c_acc = ochre_c.score(X_test, y_test)

    # Scikit-Learn
    t0 = time.perf_counter()
    sk_c = SklearnDecisionTreeClassifier(criterion="gini", max_depth=5, random_state=42)
    sk_c.fit(X_train, y_train)
    sk_c_fit = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    sk_c_preds = sk_c.predict(X_test)
    sk_c_pred = (time.perf_counter() - t0) * 1000
    sk_c_acc = accuracy_score(y_test, sk_c_preds)

    results.append({
        "Model": "DecisionTreeClassifier",
        "Ochre Fit (ms)": ochre_c_fit,
        "Sklearn Fit (ms)": sk_c_fit,
        "Fit Speedup": sk_c_fit / ochre_c_fit,
        "Ochre Predict (ms)": ochre_c_pred,
        "Sklearn Predict (ms)": sk_c_pred,
        "Ochre Metric": f"{ochre_c_acc * 100:.1f}% Acc",
        "Sklearn Metric": f"{sk_c_acc * 100:.1f}% Acc",
    })

    # 2. Decision Tree Regressor (MSE, max_depth=5)
    n_samples, n_features = 2500, 6
    np.random.seed(99)
    X = np.random.uniform(-3.0, 3.0, size=(n_samples, n_features))
    y = np.sin(X[:, 0]) + 0.5 * (X[:, 1] ** 2) - 0.2 * X[:, 2] + np.random.normal(0, 0.2, n_samples)
    X_train, X_test = X[:2000], X[2000:]
    y_train, y_test = y[:2000], y[2000:]

    # OchreML
    t0 = time.perf_counter()
    ochre_r = OchreDecisionTreeRegressor(criterion="squared_error", max_depth=5)
    ochre_r.fit(X_train, y_train)
    ochre_r_fit = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    ochre_r_preds = ochre_r.predict(X_test)
    ochre_r_pred = (time.perf_counter() - t0) * 1000
    ochre_r_score = ochre_r.score(X_test, y_test)

    # Scikit-Learn
    t0 = time.perf_counter()
    sk_r = SklearnDecisionTreeRegressor(criterion="squared_error", max_depth=5, random_state=42)
    sk_r.fit(X_train, y_train)
    sk_r_fit = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    sk_r_preds = sk_r.predict(X_test)
    sk_r_pred = (time.perf_counter() - t0) * 1000
    sk_r_score = r2_score(y_test, sk_r_preds)

    results.append({
        "Model": "DecisionTreeRegressor",
        "Ochre Fit (ms)": ochre_r_fit,
        "Sklearn Fit (ms)": sk_r_fit,
        "Fit Speedup": sk_r_fit / ochre_r_fit,
        "Ochre Predict (ms)": ochre_r_pred,
        "Sklearn Predict (ms)": sk_r_pred,
        "Ochre Metric": f"{ochre_r_score:.4f} R^2",
        "Sklearn Metric": f"{sk_r_score:.4f} R^2",
    })

    # 3. Linear Regression (Normal Equation, N=20,000)
    n_samples, n_features = 20000, 10
    np.random.seed(42)
    X = np.random.randn(n_samples, n_features)
    y = X @ np.random.uniform(1.0, 5.0, size=n_features) + 3.5 + np.random.normal(0, 0.5, n_samples)
    X_train, X_test = X[:16000], X[16000:]
    y_train, y_test = y[:16000], y[16000:]

    # OchreML
    t0 = time.perf_counter()
    ochre_l = OchreLinearRegression(fit_intercept=True)
    ochre_l.fit(X_train, y_train)
    ochre_l_fit = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    ochre_l_preds = ochre_l.predict(X_test)
    ochre_l_pred = (time.perf_counter() - t0) * 1000
    ochre_l_score = ochre_l.score(X_test, y_test)

    # Scikit-Learn
    t0 = time.perf_counter()
    sk_l = SklearnLinearRegression(fit_intercept=True)
    sk_l.fit(X_train, y_train)
    sk_l_fit = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    sk_l_preds = sk_l.predict(X_test)
    sk_l_pred = (time.perf_counter() - t0) * 1000
    sk_l_score = r2_score(y_test, sk_l_preds)

    results.append({
        "Model": "LinearRegression (20k)",
        "Ochre Fit (ms)": ochre_l_fit,
        "Sklearn Fit (ms)": sk_l_fit,
        "Fit Speedup": sk_l_fit / ochre_l_fit,
        "Ochre Predict (ms)": ochre_l_pred,
        "Sklearn Predict (ms)": sk_l_pred,
        "Ochre Metric": f"{ochre_l_score:.4f} R^2",
        "Sklearn Metric": f"{sk_l_score:.4f} R^2",
    })

    return pd.DataFrame(results)

df_benchmark = benchmark_suite()
print("=" * 80)
print(" (*≧ω≦*) OCHREML vs SCIKIT-LEARN BENCHMARK RESULTS (*≧ω≦*)")
print("=" * 80)
print(df_benchmark[["Model", "Ochre Fit (ms)", "Sklearn Fit (ms)", "Fit Speedup", "Ochre Metric", "Sklearn Metric"]].to_string(index=False))


### Visualizing Training Fit Time Comparison (o´∀｀o)


In [ ]:
models = df_benchmark["Model"]
ochre_times = df_benchmark["Ochre Fit (ms)"]
sklearn_times = df_benchmark["Sklearn Fit (ms)"]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
rects1 = ax.bar(x - width/2, ochre_times, width, label="OchreML (Pure Rust)", color="#e74c3c")
rects2 = ax.bar(x + width/2, sklearn_times, width, label="Scikit-Learn (Cython)", color="#3498db")

ax.set_ylabel("Fit Time (ms) - Lower is Faster!", fontsize=11)
ax.set_title("Training Time: OchreML vs Scikit-Learn (*≧ω≦*)", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=10)
ax.legend(frameon=True)
ax.grid(axis="y", linestyle="--", alpha=0.5)

# Attach values above bars
for rect in rects1:
    h = rect.get_height()
    ax.annotate(f"{h:.1f}ms", xy=(rect.get_x() + rect.get_width() / 2, h),
                xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=9, fontweight="bold")
for rect in rects2:
    h = rect.get_height()
    ax.annotate(f"{h:.1f}ms", xy=(rect.get_x() + rect.get_width() / 2, h),
                xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()


---
## 7. Summary & Future Roadmap ＼(＾▽＾)／

OchreML successfully couples bare-metal Rust performance with Python simplicity:
- **Faster Decision Tree Fitting:** Outperforms Scikit-Learn on decision tree training and batch prediction!
- **Accurate Analytical OLS:** Exact Normal Equation computation with zero memory overhead.
- **Universal Multi-Container Support:** Pandas, Polars, NumPy, and Lists work seamlessly.
- **Model Roadmap:** Logistic Regression, Random Forest, KNN, SVM, Naive Bayes, PCA, and K-Means coming next! (*≧ω≦*)

### Useful Links:
- **GitHub:** [github.com/arufadesuwa/ochreml](https://github.com/arufadesuwa/ochreml)
- **PyPI:** [pypi.org/project/ochreml](https://pypi.org/project/ochreml)

*Happy Machine Learning with OchreML! (*≧ω≦*)*
